In [ ]:
import sys
import os
import gc
import numpy as np
from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append('/user/leuven/384/vsc38419/0scillons/February26Oscillons/Feb26')

from core.grid import Grid
from core.rhsevolution_MG import *
from core.spacing import *
from core.display import *
from core.statevector import *
from matter.scalarmatter_MG import *

from bssn.oscillondiagnostic import get_oscillon_diagnostic
from initialdata.ModifiedGravityInitialConditions import *
from backgrounds.sphericalbackground import *

## Parameters

All physics parameters match the existing runs in `convergence_data/` (GR) and
`oscillon_MG/` (EsGB with $\lambda_{\rm GB}=0.01$).  Two **new** coupling
constants ($\lambda_{\rm GB}=0.03$ and $0.05$) will be simulated below.

In [ ]:
NOTEBOOK_DIR = "/user/leuven/384/vsc38419/0scillons/February26Oscillons/Feb26/Notebooks"
CONV_DIR     = os.path.join(NOTEBOOK_DIR, "convergence_data")
MG_DIR       = os.path.join(NOTEBOOK_DIR, "oscillon_MG")

scalar_mu = 1
selfinteraction = 0.08

TABLE_I = {
    0.04: dict(phi=-6.03334e-2, dphi=2.20256e-2, H=1.55744e-2),
    0.05: dict(phi=-7.36454e-2, dphi=2.72499e-2, H=1.92686e-2),
    0.06: dict(phi=-8.64102e-2, dphi=3.23761e-2, H=2.28934e-2),
    0.07: dict(phi=-9.86792e-2, dphi=3.74094e-2, H=2.64525e-2),
    0.08: dict(phi=-1.10495e-1, dphi=4.23540e-2, H=2.99488e-2),
    0.09: dict(phi=-1.21893e-1, dphi=4.72136e-2, H=3.33851e-2),
    0.10: dict(phi=-1.32906e-1, dphi=5.19917e-2, H=3.67637e-2),
}

u_val = TABLE_I[selfinteraction]['phi']
v_val = TABLE_I[selfinteraction]['dphi']

perturbation = -2e-2
width = 3

chi0 = 0.15
coupling = 'quadratic'

# Gauge parameters (a=b=0 for GR, modified gauge for MG)
a_mg = 0.2
b_mg = 0.4

T = 800
num_points_t = 1000
dt = T / num_points_t
t_out = np.linspace(0, T - dt, num_points_t)

# Grid (HR only)
r_max  = 150
min_dr = 1 / 16
max_dr = 2

# Coupling constants to compare
lambda_GR  = 0.0
lambda_MG  = [0.01, 0.03, 0.05]

print(f"GR  : lambda_GB = {lambda_GR}")
print(f"EsGB: lambda_GB = {lambda_MG}")

In [ ]:
my_matter = ScalarMatter(scalar_mu, selfinteraction)
my_sv     = StateVector(my_matter)

spacing = CubicSpacing(**CubicSpacing.get_parameters(r_max, min_dr, max_dr))
grid    = Grid(spacing, my_sv)
r       = grid.r
bg      = FlatSphericalBackground(r)

print(f"Grid: {r.size} points,  r in [{r[0]:.4f}, {r[-1]:.1f}]")

## 1 — Load existing GR data ($\lambda_{\rm GB}=0$)

High-resolution solution from `convergence_data/solution_HR.npy`.

In [ ]:
sol_GR = np.load(os.path.join(CONV_DIR, "solution_HR.npy"))
t_GR   = np.load(os.path.join(CONV_DIR, "t.npy"))

params_GR = (0.0, 0, 0, chi0, coupling)
osc_GR = get_oscillon_diagnostic(sol_GR, t_GR, grid, bg,
                                  ScalarMatter(scalar_mu, selfinteraction),
                                  params_GR,
                                  surface_threshold=0.05,
                                  r_max_diag=100.0)

C_max_GR = np.max(osc_GR['C'])
print(f"GR  max compactness: C = {C_max_GR:.6e}")
del sol_GR; gc.collect()

## 2 — Load existing MG data ($\lambda_{\rm GB}=0.01$)

In [ ]:
run_dir_01 = os.path.join(MG_DIR,
    f"lgb0.01_mu{selfinteraction}_a{a_mg}_b{b_mg}_amp{perturbation}_R{width}_dr{min_dr}")
print(f"Loading from: {run_dir_01}")

sol_01 = np.load(os.path.join(run_dir_01, "solution.npy"))
t_01   = np.load(os.path.join(run_dir_01, "t.npy"))

params_01 = (0.01, a_mg, b_mg, chi0, coupling)
osc_01 = get_oscillon_diagnostic(sol_01, t_01, grid, bg,
                                  ScalarMatter(scalar_mu, selfinteraction),
                                  params_01,
                                  surface_threshold=0.05,
                                  r_max_diag=100.0)

C_max_01 = np.max(osc_01['C'])
print(f"lgb=0.01  max compactness: C = {C_max_01:.6e}")
del sol_01; gc.collect()

## 3 — Run new simulations ($\lambda_{\rm GB} = 0.03,\; 0.05$)

These run at HR resolution (`min_dr = 1/16`).  Each takes a while — skip this cell
if the data already exists on disk.

In [ ]:
new_lambdas = [0.03, 0.05]

for lgb in new_lambdas:
    tag = (f"lgb{lgb}_mu{selfinteraction}_a{a_mg}_b{b_mg}"
           f"_amp{perturbation}_R{width}_dr{min_dr}")
    run_dir = os.path.join(MG_DIR, tag)
    os.makedirs(run_dir, exist_ok=True)

    sol_path = os.path.join(run_dir, "solution.npy")
    if os.path.exists(sol_path):
        print(f"\n  lgb={lgb} — data already exists, skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"Running:  lgb = {lgb}  (HR, dr = {min_dr})")
    print(f"  -> {run_dir}")

    matter_run = ScalarMatter(scalar_mu, selfinteraction)
    sv_run     = StateVector(matter_run)
    spacing_run = CubicSpacing(**CubicSpacing.get_parameters(r_max, min_dr, max_dr))
    grid_run    = Grid(spacing_run, sv_run)
    bg_run      = FlatSphericalBackground(grid_run.r)

    initial_state = get_initial_state(
        grid_run, bg_run, (lgb, a_mg, b_mg, chi0, coupling),
        matter_run, perturbation, width, scalar_mu, u_val, v_val)

    with tqdm(total=1000, unit="\u2030") as pbar:
        dense_sol = solve_ivp(
            get_rhs, [0, T], initial_state,
            args=(grid_run, bg_run, matter_run, pbar,
                  [0, T/1000], a_mg, b_mg, lgb, coupling),
            max_step=0.4 * min_dr,
            method='RK45',
            dense_output=True)

    solution = dense_sol.sol(t_out).T

    np.save(sol_path, solution)
    np.save(os.path.join(run_dir, "t.npy"), t_out)
    np.save(os.path.join(run_dir, "r.npy"), grid_run.r)
    print(f"  Saved: shape {solution.shape}")

    del dense_sol, solution, initial_state, matter_run, sv_run
    del spacing_run, grid_run, bg_run
    gc.collect()

print("\nDone.")

## 4 — Diagnostics for new MG runs

In [ ]:
osc_MG = {}      # lgb -> oscillon diagnostic dict
C_max  = {}      # lgb -> max compactness

C_max[0.0]  = C_max_GR
osc_MG[0.01] = osc_01
C_max[0.01]  = C_max_01

for lgb in new_lambdas:
    tag = (f"lgb{lgb}_mu{selfinteraction}_a{a_mg}_b{b_mg}"
           f"_amp{perturbation}_R{width}_dr{min_dr}")
    run_dir = os.path.join(MG_DIR, tag)

    sol = np.load(os.path.join(run_dir, "solution.npy"))
    t_loaded = np.load(os.path.join(run_dir, "t.npy"))

    params_mg = (lgb, a_mg, b_mg, chi0, coupling)
    osc = get_oscillon_diagnostic(sol, t_loaded, grid, bg,
                                   ScalarMatter(scalar_mu, selfinteraction),
                                   params_mg,
                                   surface_threshold=0.05,
                                   r_max_diag=100.0)

    osc_MG[lgb] = osc
    C_max[lgb]  = np.max(osc['C'])
    print(f"lgb={lgb}  max compactness: C = {C_max[lgb]:.6e}")

    del sol; gc.collect()

print(f"\nGR  reference C_max = {C_max_GR:.6e}")
for lgb in sorted(C_max):
    print(f"  lgb={lgb:6.3f}  ->  C_max = {C_max[lgb]:.6e}  "
          f"  (C/C_GR = {C_max[lgb]/C_max_GR:.3f})")

## 5 — Compactness vs $\lambda_{\rm GB}$

In [ ]:
lambdas_plot = np.array(sorted(C_max.keys()))
C_vals       = np.array([C_max[l] for l in lambdas_plot])
C_norm       = C_vals / C_max_GR

# Separate GR point from EsGB points
mg_mask = lambdas_plot > 0

fig, ax = plt.subplots(figsize=(8, 5.5))

# GR reference line (flat across all lambda)
ax.axhline(1.0, color='#4E9BD9', lw=3.5, label='GR', zorder=2)

# EsGB data
ax.plot(lambdas_plot[mg_mask], C_norm[mg_mask],
        'o-', color='#D94E2B', lw=3, ms=8, zorder=3, label='EsGB')

ax.set_xlabel(r'$\lambda_{\rm GB}$', fontsize=16)
ax.set_ylabel(r'$\mathcal{C}\;/\;\mathcal{C}_{\rm GR}$', fontsize=16)
ax.set_title('Oscillon Compactness', fontsize=16, fontweight='bold')
ax.legend(fontsize=14, loc='upper left')
ax.tick_params(labelsize=12)
ax.set_xlim(left=0)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## 6 — Compactness time series comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(osc_GR['ln_a'], osc_GR['C'] / C_max_GR,
        color='#4E9BD9', lw=2, label=r'GR ($\lambda_{\rm GB}=0$)')

colors_mg = ['#D9862B', '#D94E2B', '#8B0000']
for i, lgb in enumerate(sorted(osc_MG.keys())):
    osc = osc_MG[lgb]
    ax.plot(osc['ln_a'], osc['C'] / C_max_GR,
            color=colors_mg[i], lw=2,
            label=rf'EsGB $\lambda_{{\rm GB}}={lgb}$')

ax.set_xlabel(r'$\ln(a)$', fontsize=14)
ax.set_ylabel(r'$\mathcal{C}\;/\;\mathcal{C}_{\rm GR}^{\rm max}$', fontsize=14)
ax.set_title('Compactness evolution', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(1.5, 3.5)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()